# BGE → Qwen reranker cascade

One immutable candidate union is scored first by BGE-v2-m3. The top 30 tables per question then go to an optional Qwen3-Reranker-8B second pass. No pre-score pruning.

In [ ]:
%pip -q install -U 'sentence-transformers>=3.0' requests 'vllm==0.28.0' 'transformers>=5.0.0'

from pathlib import Path
import gc, hashlib, json, math, re, subprocess, time

RETRIEVAL_RUN = Path('/content/vifinqa_retrieval_local')
OUT = Path('/content/vifinqa_rerank_cascade')
OUT.mkdir(parents=True, exist_ok=True)
BGE_MODEL = 'BAAI/bge-reranker-v2-m3'
BGE_BATCH = 128
MAX_LENGTH = 2048
TOP_K = 30
USE_QWEN = True
candidates_path = RETRIEVAL_RUN / 'candidates.jsonl'
retrieval_manifest_path = RETRIEVAL_RUN / 'retrieval_manifest.json'
if not candidates_path.exists() or not retrieval_manifest_path.exists():
    raise FileNotFoundError(f'Missing retrieval inputs in {RETRIEVAL_RUN}')
retrieval_manifest = json.loads(retrieval_manifest_path.read_text(encoding='utf-8'))
candidate_sha = hashlib.sha256(candidates_path.read_bytes()).hexdigest()
if candidate_sha != retrieval_manifest.get('candidates_sha256'):
    raise RuntimeError('candidate hash mismatch; refusing to score')
items = [json.loads(line) for line in candidates_path.read_text(encoding='utf-8').splitlines() if line.strip()]
if len(items) != retrieval_manifest.get('questions'):
    raise RuntimeError('question count mismatch; refusing to score')
if any(not x.get('candidates') or not fact_list(x) for x in items):
    raise RuntimeError('empty candidate union or plan; refusing to score')
pair_count = sum(len(x['candidates']) * len(fact_list(x)) for x in items)
print({'questions': len(items), 'pairs': pair_count, 'candidate_sha256': candidate_sha, 'use_qwen': USE_QWEN})

In [ ]:
NUMBER = re.compile(r'^\(?[-+]?\d[\d., ]*%?\)?$')

def table_text(table):
    labels = []
    for row in table.get('rows', []):
        cells = [str(c).strip() for c in row]
        label = ' | '.join(c for c in cells if c and not NUMBER.fullmatch(c))
        if label and label not in labels:
            labels.append(label)
    return '\n'.join((
        f"{table.get('ticker', '')} {table.get('year', '')} {table.get('scope', '')}",
        str(table.get('title', '')),
        'Headers: ' + ' | '.join(map(str, table.get('header_cells', []))),
        'Rows: ' + ' ; '.join(labels),
    ))[:12000]

def fact_text(fact):
    return f"{fact.get('query', '')}\nEntity: {fact.get('entity', '')}\nPeriod: {fact.get('period', fact.get('year', ''))}\nScope: {fact.get('scope', '')}"

def fact_list(item):
    plan = item.get('plan') or item.get('verified_plan') or {}
    if plan.get('facts'):
        return plan['facts']
    metadata = item.get('metadata', {})
    entity = (metadata.get('tickers') or [''])[0]
    period = (metadata.get('years') or [None])[0]
    return [{'id': f'f{i}', 'query': str(value), 'entity': entity, 'period': period}
            for i, value in enumerate(item.get('facts', []), 1)]

def load_valid(path):
    rows = {}
    if path.exists():
        for line in path.read_text(encoding='utf-8').splitlines():
            if line.strip():
                row = json.loads(line)
                if row.get('status') == 'VALID': rows[str(row['id'])] = row
    return rows

## Stage 1 — BGE full-union scoring

BGE scores every Fact × candidate pair. Results are checkpointed per question.

In [ ]:
from sentence_transformers import CrossEncoder

encoder = CrossEncoder(BGE_MODEL, max_length=MAX_LENGTH, device='cuda', model_kwargs={'torch_dtype': 'float16'})
bge_path = OUT / 'scores_bge_v2_m3.jsonl'
done = load_valid(bge_path)
start = time.perf_counter()
with bge_path.open('a', encoding='utf-8') as handle:
    for number, item in enumerate(items, 1):
        qid = str(item['id'])
        if qid not in done:
            docs = [table_text(t) for t in item['candidates']]
            facts = fact_list(item)
            requests = [(fact_text(f), doc) for f in facts for doc in docs]
            values = [float(v) for v in encoder.predict(requests, batch_size=BGE_BATCH, convert_to_numpy=True, show_progress_bar=False)]
            if len(values) != len(requests) or not all(math.isfinite(v) for v in values): raise RuntimeError(f'invalid BGE scores for {qid}')
            width = len(docs)
            fact_scores = {f['id']: values[i * width:(i + 1) * width] for i, f in enumerate(facts)}
            row = {'id': item['id'], 'status': 'VALID', 'candidate_ids': [t['table_id'] for t in item['candidates']], 'fact_scores': fact_scores}
            done[qid] = row
            handle.write(json.dumps(row, ensure_ascii=False) + '\n'); handle.flush()
        if number % 10 == 0:
            scored = sum(len(x['candidates']) * len(fact_list(x)) for x in items[:number] if str(x['id']) in done)
            elapsed = time.perf_counter() - start
            rate = scored / max(elapsed, 1e-9)
            print(f'BGE {number}/{len(items)} | {scored}/{pair_count} pairs | {rate:.1f}/s | ETA {max(pair_count-scored,0)/max(rate,1e-9)/60:.1f} min')
bge_rows = [done[str(x['id'])] for x in items]
bge_path.write_text(''.join(json.dumps(x, ensure_ascii=False) + '\n' for x in bge_rows), encoding='utf-8')
print({'bge_score_sha256': hashlib.sha256(bge_path.read_bytes()).hexdigest()})

In [ ]:
# Aggregate BGE ranks and retain only a post-score top-30 lane for Qwen.
bge_by_id = {str(x['id']): x for x in bge_rows}
top30_path = OUT / 'bge_top30.jsonl'
top30_rows = []
for item in items:
    score_row = bge_by_id[str(item['id'])]
    aggregate = {table_id: 0.0 for table_id in score_row['candidate_ids']}
    for scores in score_row['fact_scores'].values():
        for rank, index in enumerate(sorted(range(len(scores)), key=lambda i: (-scores[i], i)), 1):
            aggregate[score_row['candidate_ids'][index]] += 1.0 / (60 + rank)
    ordered = sorted(aggregate, key=lambda table_id: (-aggregate[table_id], table_id))[:TOP_K]
    top30_rows.append({'id': item['id'], 'candidate_ids': ordered, 'bge_rrf': {k: aggregate[k] for k in ordered}})
top30_path.write_text(''.join(json.dumps(x, ensure_ascii=False) + '\n' for x in top30_rows), encoding='utf-8')
print({'questions': len(top30_rows), 'top_k': TOP_K, 'tables': sum(len(x['candidate_ids']) for x in top30_rows)})

## Stage 2 — optional Qwen second pass

Set `USE_QWEN=1` for the cascade. The BGE model is unloaded before Qwen starts so both models are never resident together.

In [ ]:
if USE_QWEN:
    import requests
    del encoder
    gc.collect()
    import torch
    torch.cuda.empty_cache()
    QWEN_MODEL = 'Qwen/Qwen3-Reranker-8B'
    QWEN_PORT = 8012
    QWEN_URL = f'http://127.0.0.1:{QWEN_PORT}/v1'
    template = '''<|im_start|>system\nJudge whether the Document contains the exact Vietnamese financial fact requested by the Query. Answer only yes or no.<|im_end|>\n<|im_start|>user\n<Query>: {{ messages | selectattr("role", "eq", "query") | map(attribute="content") | first }}\n<Document>: {{ messages | selectattr("role", "eq", "document") | map(attribute="content") | first }}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>'''
    template_path = OUT / 'qwen3_reranker.jinja'
    template_path.write_text(template, encoding='utf-8')
    overrides = json.dumps({'architectures': ['Qwen3ForSequenceClassification'], 'classifier_from_token': ['no', 'yes'], 'is_original_qwen3_reranker': True})
    def qready():
        try: return requests.get(QWEN_URL.rstrip('/') + '/models', timeout=2).ok
        except requests.RequestException: return False
    qserver = None
    if not qready():
        qserver = subprocess.Popen(['vllm', 'serve', QWEN_MODEL, '--runner', 'pooling', '--trust-remote-code', '--served-model-name', QWEN_MODEL, '--host', '127.0.0.1', '--port', str(QWEN_PORT), '--dtype', 'bfloat16', '--max-model-len', '4096', '--max-num-seqs', '128', '--max-num-batched-tokens', '65536', '--gpu-memory-utilization', '0.94', '--enable-prefix-caching', '--hf-overrides', overrides, '--chat-template', str(template_path)], stdout=(OUT / 'qwen_vllm.log').open('w'), stderr=subprocess.STDOUT)
        deadline = time.time() + 900
        while not qready():
            if qserver.poll() is not None: raise RuntimeError('Qwen vLLM exited; inspect qwen_vllm.log')
            if time.time() > deadline: raise TimeoutError('Qwen vLLM startup timed out')
            time.sleep(1)
    print({'qwen_ready': qready(), 'model': QWEN_MODEL})
else:
    print('Qwen stage disabled; BGE-only cascade output will be produced')

In [ ]:
qwen_path = OUT / 'scores_qwen8b.jsonl'
if USE_QWEN:
    qwen_done = load_valid(qwen_path)
    top30_by_id = {str(x['id']): x for x in top30_rows}
    with qwen_path.open('a', encoding='utf-8') as handle:
        for number, item in enumerate(items, 1):
            qid = str(item['id'])
            if qid not in qwen_done:
                top_ids = top30_by_id[qid]['candidate_ids']
                table_map = {t['table_id']: t for t in item['candidates']}
                docs = [table_text(table_map[k]) for k in top_ids]
                fact_scores = {}
                for fact in fact_list(item):
                    response = requests.post(QWEN_URL.rstrip('/') + '/rerank', json={'model': QWEN_MODEL, 'query': fact_text(fact), 'documents': docs, 'top_n': len(docs)}, timeout=1800)
                    response.raise_for_status()
                    values = [None] * len(docs)
                    for result in response.json().get('results', []): values[int(result['index'])] = float(result['relevance_score'])
                    if any(v is None or not math.isfinite(v) for v in values): raise RuntimeError(f'invalid Qwen scores for {qid}/{fact["id"]}')
                    fact_scores[fact['id']] = values
                row = {'id': item['id'], 'status': 'VALID', 'candidate_ids': top_ids, 'fact_scores': fact_scores}
                qwen_done[qid] = row
                handle.write(json.dumps(row, ensure_ascii=False) + '\n'); handle.flush()
            if number % 10 == 0: print('Qwen', number, '/', len(items))
    qwen_rows = [qwen_done[str(x['id'])] for x in items]
    qwen_path.write_text(''.join(json.dumps(x, ensure_ascii=False) + '\n' for x in qwen_rows), encoding='utf-8')
    print({'qwen_score_sha256': hashlib.sha256(qwen_path.read_bytes()).hexdigest()})
else:
    qwen_rows = []

## Final fused cascade output

BGE-only output is retained when Qwen is disabled. With Qwen enabled, reciprocal-rank fusion is applied inside the BGE top-30 lane.

In [ ]:
qwen_by_id = {str(x['id']): x for x in qwen_rows}
final_rows = []
for item, top in zip(items, top30_rows):
    bge_row = bge_by_id[str(item['id'])]
    bge_index = {k: i for i, k in enumerate(bge_row['candidate_ids'])}
    if USE_QWEN:
        qrow = qwen_by_id[str(item['id'])]
        fused = {k: 1.0 / (60 + i + 1) for i, k in enumerate(top['candidate_ids'])}
        for scores in qrow['fact_scores'].values():
            for rank, i in enumerate(sorted(range(len(scores)), key=lambda j: (-scores[j], j)), 1): fused[qrow['candidate_ids'][i]] += 1.0 / (60 + rank)
    else:
        fused = top['bge_rrf']
    ordered = sorted(fused, key=lambda k: (-fused[k], k))
    final_rows.append({'id': item['id'], 'candidate_ids': ordered, 'fused_score': {k: fused[k] for k in ordered}, 'model': 'BGE+Qwen8B' if USE_QWEN else 'BGE'})
final_path = OUT / 'cascade_reranked.jsonl'
final_path.write_text(''.join(json.dumps(x, ensure_ascii=False) + '\n' for x in final_rows), encoding='utf-8')
manifest = {'questions': len(items), 'input_candidate_sha256': candidate_sha, 'bge_model': BGE_MODEL, 'qwen_model': QWEN_MODEL if USE_QWEN else None, 'top_k': TOP_K, 'use_qwen': USE_QWEN, 'bge_scores_sha256': hashlib.sha256(bge_path.read_bytes()).hexdigest(), 'qwen_scores_sha256': hashlib.sha256(qwen_path.read_bytes()).hexdigest() if USE_QWEN else None, 'cascade_sha256': hashlib.sha256(final_path.read_bytes()).hexdigest()}
(OUT / 'cascade_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(manifest)

## Apply the cascade and prepare the answer-stage input
The ranking is now passed to the clean-branch `run.py`; no old ZIP is reused.

In [ ]:
import shutil
REPO_URL = 'https://github.com/lducc/vifinqa-final-submission.git'
ROOT = Path('/content/vifinqa-final-submission')
if not (ROOT / 'run.py').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(ROOT)], check=True)
RUN_PY = ROOT / 'run.py'
DATA = Path('/content/vifinqa_data')
RUN = OUT / 'clean_run'
assert RUN_PY.exists(), f'Repository clone is missing {RUN_PY}'
assert (ROOT / 'src/docs.py').exists() and (ROOT / 'scripts/validate_submission.py').exists(), 'Repository is incomplete'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ROOT)], check=True)
assert (DATA / 'questions/questions.jsonl').exists(), f'Missing organizer data: {DATA}'
ranking_path = OUT / 'cascade_ranking.json'
ranking_path.write_text(json.dumps({str(x['id']): x['candidate_ids'] for x in final_rows}, ensure_ascii=False, indent=2) + '\n')
extra = {}
for item in items:
    for table in item['candidates']:
        if table.get('table_id') and table.get('report_id'):
            extra[table['table_id']] = table['report_id']
extra_path = OUT / 'cascade_extra_tables.json'
extra_path.write_text(json.dumps(extra, ensure_ascii=False, indent=2) + '\n')
cmd = [sys.executable, str(RUN_PY), '--data-root', str(DATA), '--output-dir', str(RUN), '--ranking', str(ranking_path), '--extra-tables', str(extra_path), '--table-top-k', 'ranking', '--rerank-depth', '130', '--progress-every', '25']
if (RUN / 'rows.checkpoint.json').exists(): cmd.append('--resume')
subprocess.run(cmd, cwd=ROOT, check=True)
package = RUN / 'package'
subprocess.run([sys.executable, str(ROOT / 'scripts/validate_submission.py'), str(package)], cwd=ROOT, check=True)
answer_inputs = RUN / 'answer_inputs'
(answer_inputs / 'data/tables').mkdir(parents=True, exist_ok=True)
rows = json.loads((package / 'submission.json').read_text(encoding='utf-8'))
retrieval = [{key: row[key] for key in ('id', 'question', 'relevant_docs', 'relevant_tables', 'evidence')} for row in rows]
(answer_inputs / 'retrieval_manifest.jsonl').write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in retrieval), encoding='utf-8')
for name in {e['csv_path'] for row in retrieval for e in row['evidence']}:
    target = answer_inputs / name; target.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(package / name, target)
print({'cascade_run': str(RUN), 'answer_inputs': str(answer_inputs), 'questions': len(rows)})